In [1]:
from transformers import AutoProcessor
from PIL import Image
import sys
sys.path.insert(0,'/scratch/ywxzml3j/likaican/src/verl-qwen3-vl')
sys.path.insert(0,'/scratch/ywxzml3j/likaican/src/InSight-o3')

# processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct", use_fast=True)
processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-8B-Instruct", use_fast=True)

In [ ]:
import pickle

RESULTS_PATH = f'/scratch/ywxzml3j/likaican/mms1_rl/train_results/multi_agent_vsearch/arxiv_0307_sample_qwen3_region_loc_bbox_issue_fixed'
with open(f'{RESULTS_PATH}/global_step_149.pickle', 'rb') as f:
    results = pickle.load(f)

In [2]:
import pickle

# with open('/scratch/ywxzml3j/likaican/mms1_rl/val_results/multi_agent_vsearch/release_async_verl_latest_qwen3_vr_fix_sampling_params/o3_bench/map/trial_0/global_step_150/packet_0.pickle', 'rb') as f:
# with open('/scratch/ywxzml3j/likaican/mms1_rl/val_results/multi_agent_vsearch/release_async_max_consecutive_iou0_9_gpt_5_mini_fix_vsearcher_ans_and_pseudo_iou_and_bbox_extraction/gemini-3-flash-preview_multi-image_maxtc10/insight_doc/trial_0/global_step_150/packet_0.pickle', 'rb') as f:
# with open('/scratch/ywxzml3j/likaican/mms1_rl/val_results/multi_agent_vsearch/arxiv_0307_sample_filtered_cs_reduced_sample_50_max_pages_50-vreasoner_v2-sample_10-gpt-5-mini-insight-o3-vS-initial-rescale-0.25-prompt-refine10/my_eval/insight_doc/trial_0/global_step_0/packet_0.pickle', 'rb') as f:
# with open('/scratch/ywxzml3j/likaican/mms1_rl/val_results/multi_agent_vsearch/arxiv_0307_sample_qwen3_region_loc_bbox_issue_fixed/insight_doc/trial_0/global_step_150/packet_6.pickle', 'rb') as f:
# with open('/scratch/ywxzml3j/likaican/mms1_rl/val_results/multi_agent_vsearch/arxiv_0307_sample_qwen3_region_loc_bbox_issue_fixed/my_eval/insight_doc/trial_0/global_step_150/packet_1.pickle', 'rb') as f:
# with open('/scratch/ywxzml3j/likaican/mms1_rl/val_results/multi_agent_vsearch/arxiv_0307_sample_filtered_cs_reduced_sample_50_max_pages_50-vreasoner_v2-sample_10-gpt-5-mini-qwen3-initial-rescale-0.25-prompt-refine10-correct_image_process/my_eval/insight_doc/trial_0/global_step_0/packet_9.pickle', 'rb') as f:
# with open('/scratch/ywxzml3j/likaican/mms1_rl/val_results/multi_agent_vsearch/arxiv_0307_sample_qwen3_region_loc_bbox_issue_fixed/veqa_batch_0350_mveqa_batch_0352_sample_50_maxp40_simple_prompt/insight_doc_0352/trial_0/global_step_150/packet_3.pickle', 'rb') as f:
# with open('/scratch/ywxzml3j/likaican/mms1_rl/val_results/multi_agent_vsearch/arxiv_0307_sample_qwen3_region_loc_bbox_issue_fixed/veqa_batch_0350_mveqa_batch_0352_sample_50_maxp40_simple_prompt_answer_tag/insight_doc_0352/trial_0/global_step_150/packet_6.pickle', 'rb') as f:
with open('/scratch/ywxzml3j/likaican/temp/iq_base_eval_default_sampling_insight_rl15360_r7/val_results/insight_doc/iq_base_eval_default_sampling_insight_rl15360_r7/heldout/insight_doc_0352/trial_0/global_step_0/packet_0.pickle', 'rb') as f:
    results = pickle.load(f)

In [3]:
import pandas as pd
df = pd.DataFrame({key: results.non_tensor_batch.get(key) for key in ('data_source', 'score', 'format_reward', 'n_tool_calls', 'failure_reasons', 'agent_name', 'job_id', 'parent_job_id', 'idx_as_child')})
parent_indices = []
job_id_to_idx = {jid: idx for idx, jid in enumerate(df['job_id'])}
for parent_id in df['parent_job_id']:
    if pd.isnull(parent_id):
        parent_indices.append(-1)
    else:
        parent_indices.append(job_id_to_idx.get(parent_id, -1))
df['parent_index'] = parent_indices
df.drop(columns=['parent_job_id', 'job_id'], inplace=True)
df

,data_source,score,format_reward,n_tool_calls,failure_reasons,agent_name,idx_as_child,parent_index
0,insight_doc_0352,0.0,1.0,None,None,insight_qwen_agent,None,-1


In [4]:
# Gather the child indices for each job by building a reverse map from parent_index to child indices
from collections import defaultdict

children_indices = defaultdict(list)
for idx, parent_idx in enumerate(df['parent_index']):
    if parent_idx != -1:
        children_indices[parent_idx].append(idx)

for parent_idx, children_idxs in children_indices.items():
    children_idxs.sort(key=lambda idx: df.iloc[idx]['idx_as_child'])

children_indices

defaultdict(list, {})

In [5]:
from IPython.display import Pretty, Markdown, display


OMIT_SYSTEM_PROMPT = True
OMIT_SUBLEVEL_INPUT_IMAGE = True


def display_conversation(idx, level=0, show_sublevel=True, image_placeholder_text='<|vision_start|><|vision_end|>'):
    image_cnt = 0
    child_cnt = 0

    def display_next_image(skip=False):
        nonlocal image_cnt
        multi_modal_data = results.non_tensor_batch['multi_modal_data'][idx]
        if 'image' in multi_modal_data:
            image = multi_modal_data['image'][image_cnt]
        elif 'images' in multi_modal_data:
            image = multi_modal_data['images'][image_cnt]
        else:
            raise ValueError(f"Unknown multi_modal_data: {multi_modal_data}")
        if not skip:
            display(image)
        else:
            display(Pretty("[image omitted]"))
        print('image size:', image.size)
        image_cnt += 1

    first_user_message_passed = False

    display(Markdown(f"`[{results.non_tensor_batch['agent_name'][idx]}]`"))

    for message in results.non_tensor_batch['messages'][idx]:
        if show_sublevel and message['role'] not in ('assistant', 'system') and first_user_message_passed:
            # assuming all messages that are not from assistant or system are tool responses
            display(Markdown(f"---"))
            indices = children_indices[idx]
            display_conversation(indices[child_cnt], level=level+1, show_sublevel=False)
            display(Markdown(f"---"))
            child_cnt += 1
            display(Markdown(f"`[{results.non_tensor_batch['agent_name'][idx]}]`"))

        skip_image = OMIT_SUBLEVEL_INPUT_IMAGE and level and not first_user_message_passed

        if message['role'] == 'user':
            first_user_message_passed = True

        display(Markdown(f"**{message['role'].capitalize()}:**"))

        if OMIT_SYSTEM_PROMPT and message['role'] == 'system':
            display(Pretty("[system prompt omitted]"))
            continue

        # Convert content to a list if it's not
        contents = message['content']
        if not isinstance(message['content'], list):
            contents = [contents]

        # Display each content
        for content in contents:
            if isinstance(content, str):
                content = {'type': 'text', 'text': content}

            if content['type'] == 'text':
                if not content['text']:
                    continue
                chunks = content['text'].split(image_placeholder_text)  
                for i, chunk in enumerate(chunks):
                    display(Pretty(f"{chunk}"))
                    if i < len(chunks) - 1:
                        display_next_image(skip=skip_image)

            elif content['type'] == 'image':
                display_next_image(skip=skip_image)
                
            else:
                raise ValueError(f"Unknown content type: {content['type']}")


In [6]:
results.non_tensor_batch

{'data_source': array(['insight_doc_0352'], dtype=object),
 'reward_model': array([{'ground_truth': 'The total combined thickness of the skin layer (1 mm) and the depth layer (6.00 mm) is 7.00 mm.', 'style': 'rule'}],
       dtype=object),
 'uid': array(['ef9399bc-91fe-42e8-93a3-84da8a0f80fb'], dtype=object),
 '__num_turns__': array([14], dtype=int32),
 'multi_modal_inputs': array([None], dtype=object),
 'insight_original_images': array([list([<PIL.Image.Image image mode=RGB size=1708x2212 at 0x155334D860C0>, <PIL.Image.Image image mode=RGB size=1708x2212 at 0x155334D86120>, <PIL.Image.Image image mode=RGB size=1708x2212 at 0x155334D86180>, <PIL.Image.Image image mode=RGB size=1708x2212 at 0x155334D861E0>, <PIL.Image.Image image mode=RGB size=1708x2212 at 0x155334D86240>, <PIL.Image.Image image mode=RGB size=1708x2212 at 0x155334D862A0>, <PIL.Image.Image image mode=RGB size=1708x2212 at 0x155334D86300>])],
       dtype=object),
 'tool_rewards': array([list([])], dtype=object),
 'respon

In [7]:
idx = 0
display_conversation(idx, show_sublevel=(idx in children_indices))
display(Markdown("---"))
print('Ground truth:', results.non_tensor_batch['reward_model'][idx].get('ground_truth'))
print('Extracted answer:', results.non_tensor_batch['reward_model'][idx].get('extracted_answer'))
print('Score:', results.non_tensor_batch['score'][idx])

`[insight_qwen_agent]`

**System:**

[system prompt omitted]

**User:**

Image 0:

KeyError: 'multi_modal_data'